# When a bird map looks empty: occurrence records are not absence

**Jihun Oh | EDS Book idea #14 | preparation draft, NOT peer-reviewed or accepted. The offline teaching path was author-tested on Binder at commit 3ae6a75 (27 October 2026); a DOI-backed release remains unverified.**

## Purpose
Compare analyst-chosen spatial grids and explain how quality criteria affect mapped *occurrence records*. KoreAves is the author's motivation, **not** the underlying source dataset. The method is transferable to other countries after independent local checks.

**Learning objectives:** Distinguish reported location uncertainty from grid size; count records rather than birds; see that empty cells cannot prove a species absent; disclose source and sampling limitations.


## Data and attribution
The live GBIF occurrence search API is used for South Korean human bird observations in **2024**, constrained to CC0 records with coordinates. This query takes only the **first 100 API results**. It is **non-random**, **mutable**, not a time series, and **has no occurrence-download DOI**. It is an *illustrative preview*, not a fixed citable sample for publication. Original datasetKey values remain visible for attribution; data and imagery licences may differ. Before review, use a stable, citable [GBIF download DOI](https://www.gbif.org/citation-guidelines). No third-party media are copied.


In [ ]:
from collections import Counter
from urllib.parse import urlencode
from urllib.request import Request,urlopen
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pyproj import Transformer
from occurrence_grid import screen,aggregate
from grid_snapshot import load_snapshot,derive_grids

# Offline/aggregated by default; live first-page records are opt-in.
LIVE_GBIF = False
snapshot=load_snapshot('data/first-page-grids-20261026.json')
query={'country':'KR','taxonKey':'212','basisOfRecord':'HUMAN_OBSERVATION','year':'2024',
       'license':'CC0_1_0','hasCoordinate':'true','hasGeospatialIssue':'false',
       'limit':'100','offset':'0'}
url='https://api.gbif.org/v1/occurrence/search?'+urlencode(query)
if LIVE_GBIF:
    with urlopen(Request(url,headers={'User-Agent':'EDSBook-records-education/0.2'}),timeout=30) as response:
        result=json.load(response)
    records=result['results']
    assert 0<len(records)<=100
    publisher_keys=Counter(str(r.get('datasetKey') or '') for r in records)
    unknown=set(publisher_keys)-set(snapshot['dataset_dois'])
    if unknown:
        raise ValueError('New dataset keys need manual licence/citation review: '+str(unknown))
    print('LIVE first page:',len(records),'of',result.get('count'))
else:
    print('OFFLINE derived grid snapshot from:',snapshot['retrieved_at_utc'])
    print('No individual coordinates or occurrence IDs are redistributed.')
    print('Source first page:',snapshot['rows_in_page'],'of',snapshot['api_total_at_retrieval'])
    print('Publisher source counts:',snapshot['first_page_dataset_counts'])
print('GBIF selection query:',url)
print('Dataset DOIs (NOT a citable versioned occurrence download):',snapshot['dataset_dois'])


### Dataset provenance and licence hierarchy

This live-query preview encounters two publisher datasets, audited from the GBIF dataset registry on 26 October 2026:

- iNaturalist Research-grade Observations ([dataset DOI 10.15468/ab3s5x](https://doi.org/10.15468/ab3s5x)). The overall published dataset metadata indicates CC BY-NC 4.0. Some individual occurrence records, including those selected by the query, are CC0. This does **not** grant permission to reuse attached photographs or audio.
- 2024 Recording Local Biodiversity NSMK ([dataset DOI 10.15468/ya2fjz](https://doi.org/10.15468/ya2fjz)). Dataset-level metadata indicates CC0 1.0.

These **dataset DOIs are not the selection-specific GBIF occurrence DOWNLOAD DOI**. The source query can change as the GBIF index updates; all counts shown are time-dependent. The notebook publishes neither precise coordinates nor third-party media. It needs a separately created, stable GBIF occurrence download and proper publisher citation before editorial review.


## Quality screening
Exclude missing/invalid coordinate uncertainty and uncertainty values above **1,000 metres**, an *illustrative exercise threshold*, NOT a biodiversity quality standard. Values missing from the metadata do **not** imply perfect accuracy. Duplicate GBIF IDs are counted once. The selection still has taxonomic, access and observer-effort bias.


In [ ]:
if LIVE_GBIF:
    accepted,exclusions=screen(records,threshold=1000)
    accepted_count=len(accepted)
    print('LIVE accepted records:',accepted_count)
    print('LIVE screening exclusions:',dict(sorted(exclusions.items())))
    assert accepted_count+sum(exclusions.values())==len(records)
    print('LIVE accepted publisher:',dict(Counter(x['datasetKey'] for x in accepted)))
else:
    accepted_count=snapshot['accepted_count']
    exclusions=snapshot['excluded']
    print('FROZEN screened records:',accepted_count)
    print('FROZEN screening exclusions:',exclusions)
    print('Accepted publisher:',snapshot['accepted_dataset_counts'])
    assert accepted_count+sum(exclusions.values())==snapshot['rows_in_page']
if len(snapshot['accepted_dataset_counts'])==1:
    print('SOURCE SELECTION BIAS: One publisher supplies all frozen accepted records.')


## Map two spatial aggregation scales
Reproject longitude/latitude (WGS84) into **WGS84 / UTM zone 52N**, a metric projection appropriate for this *illustrative* Korean example. UTM zone selection should be reassessed outside the study area. **10 km** and **50 km** are analyst-defined bin sizes, not the input data's native precision. The maps draw only cells containing qualifying records, within an extent determined by those records. Uncolored space is not a surveyed zero or evidence of biological absence: the example has no systematic sampling-coverage layer.


In [ ]:
if LIVE_GBIF:
    if not accepted:
        raise ValueError('No qualifying records; do not relax quality criteria')
    transform=Transformer.from_crs('EPSG:4326','EPSG:32652',always_xy=True)
    xy=[transform.transform(x['lon'],x['lat']) for x in accepted]
    fine=aggregate(xy,10000)
    coarse=aggregate(xy,50000)
else:
    fine,coarse=derive_grids(snapshot['cells_10km'])
assert sum(fine.values())==sum(coarse.values())==accepted_count
print('10-km occupied cells:',len(fine),'50-km occupied cells:',len(coarse),'screened records:',accepted_count)


In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
# Shared extent and colour mapping keep both map scales comparable.
max_count=max(list(fine.values())+list(coarse.values()))
norm=Normalize(vmin=0,vmax=max_count)
cmap=plt.get_cmap('YlGnBu')
xmin=min(min(gx*step for gx,gy in counts) for step,counts in ((10000,fine),(50000,coarse)))
xmax=max(max((gx+1)*step for gx,gy in counts) for step,counts in ((10000,fine),(50000,coarse)))
ymin=min(min(gy*step for gx,gy in counts) for step,counts in ((10000,fine),(50000,coarse)))
ymax=max(max((gy+1)*step for gx,gy in counts) for step,counts in ((10000,fine),(50000,coarse)))
fig,axs=plt.subplots(1,2,figsize=(12,6),sharex=True,sharey=True)
for ax,(label,step,counts) in zip(axs,[('10 km analytical grid',10000,fine),('50 km analytical grid',50000,coarse)]):
    for (gx,gy),count in counts.items():
        ax.add_patch(Rectangle((gx*step/1000,gy*step/1000),step/1000,step/1000,
                               facecolor=cmap(norm(count)),edgecolor='#26465d',linewidth=0.65))
        if step==50000:
            ax.text((gx+0.5)*step/1000,(gy+0.5)*step/1000,str(count),
                    ha='center',va='center',fontsize=9,
                    color='white' if norm(count)>0.5 else '#102a42')
    ax.set_xlim((xmin-15000)/1000,(xmax+15000)/1000)
    ax.set_ylim((ymin-15000)/1000,(ymax+15000)/1000)
    ax.set_aspect('equal',adjustable='box')
    ax.set_title(f'{label} | {len(counts)} occupied cells',fontsize=11,pad=9)
    ax.set_xlabel('Easting, UTM 52N (km)')
    ax.set_ylabel('Northing, UTM 52N (km)')
    ax.grid(True,alpha=0.18)
cax=fig.add_axes([0.90,0.25,0.018,0.50])
fig.colorbar(ScalarMappable(norm=norm,cmap=cmap),cax=cax,
             label='Accepted occurrence records per occupied cell')
fig.suptitle('Analyst grid size changes mapped record patterns, not population abundance',
             fontsize=13,y=0.96)
fig.text(0.48,0.035,'GBIF CC0 individual records, 2024, Korea, first 100 search rows, illustrative 1 km uncertainty screen',
         ha='center',fontsize=8,color='#394d5f')
fig.subplots_adjust(top=0.84,bottom=0.17,left=0.08,right=0.82,wspace=0.28)
plt.show()


## Interpret: claim, evidence, uncertainty

**Supported:** For the current non-random API first page and declared exclusions, an occupied cell contains at least one accepted observation record.

**Unsupported:** A blank cell proves the bird absent; more records show greater bird abundance; this map predicts where a visitor will see a bird; different cell counts constitute a population trend.

**What would strengthen such claims?** Citable versioned primary source, accurate taxon matching, appropriate sampling frame, surveyed zeroes, observer-effort information, seasonal context, and independently confirmed public site access. The 10-km/50-km view does not resolve those issues.

**Exercise:** Change one analyst grid size. State one visual inference that changes, one that remains valid, and one that is unsupported in both maps.


## Summary and next steps

- Mapped accepted *records* are not birds, ranges or biological absences.
- Missing uncertainty is unknown and is disclosed as an exclusion.
- Grid size and source-coordinate uncertainty are different quantities.
- Before requesting EDS Book review: create a versioned, DOI-citable occurrence dataset; reconcile all publisher credits/licences; rerun across the intended geography; retest the final DOI-backed version on Binder; then provide the editor the PR evidence. The earlier Binder check covered only the default offline aggregate.

**Original data access:** [GBIF](https://www.gbif.org/citation-guidelines). **Code:** Python; upstream official EDS Book template retained. **Contact:** comments in [idea #14](https://github.com/eds-book/notebooks-ideas/issues/14).
